# ISIC 2017 Binary Classification with ViT-Tiny and Attention Rollout

This notebook evaluates a pretrained ViT-Tiny model for binary melanoma classification on the official ISIC 2017 train, validation, and test splits. It compares a head-only baseline with full fine-tuning and quantifies attention alignment against lesion masks using last-layer CLS attention and attention rollout.

### Recorded results

- **Head-only test performance:** accuracy 71.67%, F1-score 32.00%, AUC 65.69%.
- **After full fine-tuning:** accuracy 83.50%, F1-score 53.95%, AUC 82.00%.
- **All-test attention Dice:** last-layer CLS attention improved from 0.2862 to 0.3875; attention rollout improved from 0.2518 to 0.3458.

> Notebook outputs and execution counters are intentionally cleared. Configure the Google Drive dataset and output paths before running the notebook in Google Colab.


## Environment, Imports, and Configuration


In [ ]:
!pip -q install timm opencv-python scikit-learn

import os
import re
import gc
import glob
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ISIC_ROOT = "/content/drive/MyDrive/datasets/ISIC_2017"

SAVE_DIR = "/content/drive/MyDrive/isic2017_vit_tiny_attention_rollout_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0

CLASS_NAMES = ["non_melanoma", "melanoma"]
NUM_CLASSES = 2
MELANOMA_CLASS = 1

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def trusted_torch_load(path, map_location=None):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)

print("Device:", DEVICE)
print("ISIC root exists:", os.path.exists(ISIC_ROOT))
print("Save dir:", SAVE_DIR)


## Dataset Discovery and Indexing


In [ ]:
csv_candidates = glob.glob(
    os.path.join(ISIC_ROOT, "**", "*.csv"),
    recursive=True
)

isic2017_csvs = []

for p in csv_candidates:
    pl = p.lower()
    if (
        "groundtruth" in pl
        or "classification" in pl
        or "truth" in pl
    ):
        isic2017_csvs.append(p)

print("Candidate CSVs:")
for p in isic2017_csvs:
    print(p)

assert len(isic2017_csvs) > 0, "No ISIC 2017 CSV files were found."

def detect_split_from_path(path):
    pl = path.lower()
    if "train" in pl or "training" in pl:
        return "train"
    if "valid" in pl or "validation" in pl:
        return "val"
    if "test" in pl:
        return "test"
    return None

split_csv = {}

for p in isic2017_csvs:
    split = detect_split_from_path(p)
    if split is not None and split not in split_csv:
        split_csv[split] = p

print("\nDetected CSV splits:")
print(split_csv)

assert "train" in split_csv, "The training CSV was not found."
assert "val" in split_csv, "The validation CSV was not found."
assert "test" in split_csv, "The test CSV was not found."

# Index images and masks

image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

all_image_files = []

for root, dirs, files in os.walk(ISIC_ROOT):
    for fname in files:
        if fname.lower().endswith(image_exts):
            all_image_files.append(os.path.join(root, fname))

image_index = {}
mask_index = {}

for path in all_image_files:
    fname = os.path.basename(path)
    match = re.search(r"(ISIC_\d+)", fname, re.IGNORECASE)

    if match is None:
        continue

    image_id = match.group(1).upper()
    pl = path.lower()

    is_mask = (
        "segmentation" in pl
        or "mask" in pl
        or "part1_groundtruth" in pl
    )

    if is_mask:
        if image_id not in mask_index:
            mask_index[image_id] = path
    else:
        if image_id not in image_index:
            image_index[image_id] = path

print("Indexed images:", len(image_index))
print("Indexed masks:", len(mask_index))


## Metadata Construction


In [ ]:
def load_isic2017_split(csv_path, split_name):
    df = pd.read_csv(csv_path)
    df.columns = [str(c).strip() for c in df.columns]

    image_col = None
    for c in df.columns:
        if "image" in c.lower() or "id" in c.lower():
            image_col = c
            break

    if image_col is None:
        image_col = df.columns[0]

    melanoma_col = None
    for c in df.columns:
        if "melanoma" in c.lower():
            melanoma_col = c
            break

    assert melanoma_col is not None, f"No melanoma column was found in {csv_path}"

    rows = []

    for _, row in df.iterrows():
        image_id = str(row[image_col]).strip().upper()
        image_id = image_id.replace(".JPG", "").replace(".JPEG", "").replace(".PNG", "")

        if image_id not in image_index:
            continue

        if image_id not in mask_index:
            continue

        label = int(float(row[melanoma_col]))

        rows.append({
            "image_id": image_id,
            "image_path": image_index[image_id],
            "mask_path": mask_index[image_id],
            "binary_label": label,
            "binary_name": CLASS_NAMES[label],
            "split": split_name
        })

    return pd.DataFrame(rows)

train_df = load_isic2017_split(split_csv["train"], "train")
val_df   = load_isic2017_split(split_csv["val"], "val")
test_df  = load_isic2017_split(split_csv["test"], "test")

print("Train:", len(train_df))
display(train_df["binary_name"].value_counts())

print("Validation:", len(val_df))
display(val_df["binary_name"].value_counts())

print("Test:", len(test_df))
display(test_df["binary_name"].value_counts())

train_df.to_csv(os.path.join(SAVE_DIR, "isic2017_train_metadata.csv"), index=False)
val_df.to_csv(os.path.join(SAVE_DIR, "isic2017_val_metadata.csv"), index=False)
test_df.to_csv(os.path.join(SAVE_DIR, "isic2017_test_metadata.csv"), index=False)

display(train_df.head())


## Dataset and Data Loaders


In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

class ISIC2017Dataset(Dataset):
    def __init__(self, dataframe, transform=None, return_mask=False):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.return_mask = return_mask

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_pil = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image_tensor = self.transform(image_pil)
        else:
            image_tensor = image_pil

        label = torch.tensor(int(row["binary_label"]), dtype=torch.long)

        if self.return_mask:
            image_np = np.array(
                image_pil.resize((IMG_SIZE, IMG_SIZE))
            ).astype(np.float32) / 255.0

            mask_pil = Image.open(row["mask_path"]).convert("L").resize((IMG_SIZE, IMG_SIZE))
            mask_np = np.array(mask_pil)
            mask_np = (mask_np > 127).astype(np.uint8)

            return image_tensor, image_np, mask_np, label, row["image_id"]

        return image_tensor, label, row["image_id"], row["image_path"], row["mask_path"]

train_ds = ISIC2017Dataset(train_df, transform=train_tfms, return_mask=False)
val_ds   = ISIC2017Dataset(val_df, transform=eval_tfms, return_mask=False)
test_ds  = ISIC2017Dataset(test_df, transform=eval_tfms, return_mask=False)

isic_mask_test_ds = ISIC2017Dataset(test_df, transform=eval_tfms, return_mask=True)

class_counts = train_df["binary_label"].value_counts().sort_index()

sample_weights = train_df["binary_label"].map(
    lambda x: 1.0 / class_counts[x]
).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Dataloaders ready.")


## ViT-Tiny Model


In [ ]:
def create_vit_tiny(num_classes=2, pretrained=True):
    model = timm.create_model(
        "vit_tiny_patch16_224",
        pretrained=pretrained,
        num_classes=num_classes
    )

    # Important for attention extraction in timm
    for block in model.blocks:
        if hasattr(block.attn, "fused_attn"):
            block.attn.fused_attn = False

    return model

def freeze_vit_backbone(model):
    for name, param in model.named_parameters():
        param.requires_grad = False

    # classification head
    for param in model.head.parameters():
        param.requires_grad = True

def unfreeze_vit_all(model):
    for param in model.parameters():
        param.requires_grad = True

model = create_vit_tiny(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)

print(f"Model ready: {model.__class__.__name__}")


## Loss and Evaluation


In [ ]:
class_weights = np.zeros(NUM_CLASSES, dtype=np.float32)

for c in range(NUM_CLASSES):
    class_weights[c] = len(train_df) / max((train_df["binary_label"] == c).sum(), 1)

class_weights = class_weights / class_weights.mean()

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
)

print("Class weights:", class_weights)

@torch.no_grad()
def evaluate_model(model, loader, threshold=0.5):
    model.eval()

    total_loss = 0.0
    n = 0

    all_probs = []
    all_preds = []
    all_true = []
    all_ids = []
    all_paths = []
    all_masks = []

    for images, labels, image_ids, image_paths, mask_paths in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        probs = F.softmax(logits, dim=1)[:, 1]
        preds = (probs >= threshold).long()

        bs = images.size(0)
        total_loss += loss.item() * bs
        n += bs

        all_probs.extend(probs.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_true.extend(labels.detach().cpu().numpy().tolist())
        all_ids.extend(list(image_ids))
        all_paths.extend(list(image_paths))
        all_masks.extend(list(mask_paths))

    y_true = np.array(all_true)
    y_prob = np.array(all_probs)
    y_pred = np.array(all_preds)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    specificity = tn / (tn + fp + 1e-8)

    return {
        "loss": total_loss / max(n, 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred,
        "image_ids": all_ids,
        "image_paths": all_paths,
        "mask_paths": all_masks
    }

def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 91)

    best_thr = 0.5
    best_f1 = -1

    for thr in thresholds:
        preds = (y_prob >= thr).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    return float(best_thr), float(best_f1)


## Training Function


In [ ]:
def train_phase(
    model,
    phase_name,
    train_loader,
    val_loader,
    epochs,
    lr,
    save_path
):
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=1e-4
    )

    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    best_val_auc = -1
    best_threshold = 0.5
    history = []

    for epoch in range(1, epochs + 1):
        model.train()

        running_loss = 0.0
        n = 0

        for images, labels, _, _, _ in tqdm(train_loader, leave=False, desc=f"{phase_name} epoch {epoch}"):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            bs = images.size(0)
            running_loss += loss.item() * bs
            n += bs

        train_loss = running_loss / max(n, 1)

        val_raw = evaluate_model(model, val_loader, threshold=0.5)
        best_thr, _ = find_best_threshold(val_raw["y_true"], val_raw["y_prob"])
        val_metrics = evaluate_model(model, val_loader, threshold=best_thr)

        row = {
            "phase": phase_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_threshold": best_thr,
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall_sensitivity": val_metrics["recall_sensitivity"],
            "val_specificity": val_metrics["specificity"],
            "val_f1": val_metrics["f1"],
            "val_auc": val_metrics["auc"],
            "val_tn": val_metrics["tn"],
            "val_fp": val_metrics["fp"],
            "val_fn": val_metrics["fn"],
            "val_tp": val_metrics["tp"]
        }

        history.append(row)

        print(
            f"{phase_name} | Epoch {epoch}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_auc={val_metrics['auc']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f} | "
            f"thr={best_thr:.2f} | "
            f"recall={val_metrics['recall_sensitivity']:.4f}"
        )

        if val_metrics["auc"] > best_val_auc:
            best_val_auc = val_metrics["auc"]
            best_threshold = best_thr

            torch.save({
                "phase": phase_name,
                "epoch": epoch,
                "model_state_dict": copy.deepcopy(model.state_dict()),
                "val_auc": best_val_auc,
                "threshold": best_threshold,
                "class_names": CLASS_NAMES
            }, save_path)

            print("Saved best:", save_path)

    return pd.DataFrame(history)


## Head-Only Training and Full Fine-Tuning


In [ ]:
FORCE_RETRAIN = True

before_ckpt_path = os.path.join(SAVE_DIR, "vit_tiny_isic2017_before_headonly.pth")
after_ckpt_path  = os.path.join(SAVE_DIR, "vit_tiny_isic2017_after_fullfinetune.pth")

if FORCE_RETRAIN:
    for p in [before_ckpt_path, after_ckpt_path]:
        if os.path.exists(p):
            os.remove(p)
            print("Removed old checkpoint:", p)

model = create_vit_tiny(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)

history_parts = []

# Phase 1: head-only

freeze_vit_backbone(model)

history_head = train_phase(
    model=model,
    phase_name="Before FT - head only",
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,
    lr=1e-3,
    save_path=before_ckpt_path
)

history_parts.append(history_head)

assert os.path.exists(before_ckpt_path), "Head-only checkpoint was not created."

# Phase 2: full fine-tuning

before_ckpt = trusted_torch_load(before_ckpt_path, map_location=DEVICE)
model.load_state_dict(before_ckpt["model_state_dict"])

unfreeze_vit_all(model)

history_full = train_phase(
    model=model,
    phase_name="After FT - full fine-tuning",
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=10,
    lr=1e-5,
    save_path=after_ckpt_path
)

history_parts.append(history_full)

assert os.path.exists(after_ckpt_path), "Full fine-tuning checkpoint was not created."

history_all = pd.concat(history_parts, ignore_index=True)

history_path = os.path.join(SAVE_DIR, "isic2017_vit_tiny_training_history.csv")
history_all.to_csv(history_path, index=False)

display(history_all)

print("Saved history:", history_path)
print("Before checkpoint:", before_ckpt_path, os.path.exists(before_ckpt_path))
print("After checkpoint:", after_ckpt_path, os.path.exists(after_ckpt_path))


## Test Evaluation Before and After Fine-Tuning


In [ ]:
def load_vit_checkpoint(path):
    model = create_vit_tiny(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
    ckpt = trusted_torch_load(path, map_location=DEVICE)

    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    for block in model.blocks:
        if hasattr(block.attn, "fused_attn"):
            block.attn.fused_attn = False

    threshold = ckpt.get("threshold", 0.5)

    return model, threshold, ckpt

model_before, thr_before, ckpt_before = load_vit_checkpoint(before_ckpt_path)
model_after, thr_after, ckpt_after = load_vit_checkpoint(after_ckpt_path)

print("Before threshold:", thr_before)
print("After threshold:", thr_after)

test_before = evaluate_model(model_before, test_loader, threshold=thr_before)
test_after  = evaluate_model(model_after, test_loader, threshold=thr_after)

test_metrics_df = pd.DataFrame([
    {
        "Dataset": "ISIC2017",
        "Model": "ViT-Tiny",
        "Stage": "Before FT - head only",
        "Threshold": thr_before,
        "Accuracy": test_before["accuracy"],
        "Precision": test_before["precision"],
        "Recall/Sensitivity": test_before["recall_sensitivity"],
        "Specificity": test_before["specificity"],
        "F1": test_before["f1"],
        "AUC": test_before["auc"],
        "TN": test_before["tn"],
        "FP": test_before["fp"],
        "FN": test_before["fn"],
        "TP": test_before["tp"]
    },
    {
        "Dataset": "ISIC2017",
        "Model": "ViT-Tiny",
        "Stage": "After FT - full fine-tuning",
        "Threshold": thr_after,
        "Accuracy": test_after["accuracy"],
        "Precision": test_after["precision"],
        "Recall/Sensitivity": test_after["recall_sensitivity"],
        "Specificity": test_after["specificity"],
        "F1": test_after["f1"],
        "AUC": test_after["auc"],
        "TN": test_after["tn"],
        "FP": test_after["fp"],
        "FN": test_after["fn"],
        "TP": test_after["tp"]
    }
])

test_metrics_path = os.path.join(
    SAVE_DIR,
    "table_isic2017_vit_tiny_classification_before_after.csv"
)

test_metrics_df.to_csv(test_metrics_path, index=False)

display(test_metrics_df)
print("Saved:", test_metrics_path)

test_pred_df = test_df.copy()
test_pred_df["true_label"] = test_after["y_true"]
test_pred_df["prob_before"] = test_before["y_prob"]
test_pred_df["pred_before"] = test_before["y_pred"]
test_pred_df["prob_after"] = test_after["y_prob"]
test_pred_df["pred_after"] = test_after["y_pred"]

pred_path = os.path.join(SAVE_DIR, "isic2017_vit_tiny_test_predictions_before_after.csv")
test_pred_df.to_csv(pred_path, index=False)

print("Saved predictions:", pred_path)


## Attention Extraction Utilities


In [ ]:
ATTN_DIR = os.path.join(SAVE_DIR, "attention_rollout_outputs")
os.makedirs(ATTN_DIR, exist_ok=True)

def normalize_map(x):
    x = np.nan_to_num(x)
    x = x - np.min(x)
    x = x / (np.max(x) + 1e-8)
    return x

def overlay_heatmap(image_np, heatmap, alpha=0.45):
    heatmap = normalize_map(heatmap)
    heatmap_uint8 = np.uint8(255 * heatmap)

    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    heatmap_color = heatmap_color.astype(np.float32) / 255.0

    overlay = (1 - alpha) * image_np + alpha * heatmap_color
    overlay = np.clip(overlay, 0, 1)

    return overlay

def top_percent_binary_map(attr_map, top_percent=20):
    attr_map = normalize_map(attr_map)
    threshold = np.percentile(attr_map, 100 - top_percent)
    return (attr_map >= threshold).astype(np.uint8)

def mask_alignment_metrics(attr_map, mask, top_percent=20):
    attr_map = normalize_map(attr_map)
    mask = (mask > 0).astype(np.uint8)

    binary_attr = top_percent_binary_map(attr_map, top_percent=top_percent)

    intersection = np.logical_and(binary_attr == 1, mask == 1).sum()
    union = np.logical_or(binary_attr == 1, mask == 1).sum()

    iou = intersection / (union + 1e-8)

    dice = (2 * intersection) / (
        binary_attr.sum() + mask.sum() + 1e-8
    )

    attr_sum = attr_map.sum() + 1e-8
    inside = attr_map[mask == 1].sum() / attr_sum
    outside = attr_map[mask == 0].sum() / attr_sum

    return {
        "IoU": float(iou),
        "Dice": float(dice),
        "Inside-Lesion Attribution": float(inside),
        "Outside-Lesion Leakage": float(outside)
    }

@torch.no_grad()
def predict_vit_image(model, image_tensor, threshold=0.5):
    model.eval()
    x = image_tensor.unsqueeze(0).to(DEVICE)

    logits = model(x)
    probs = F.softmax(logits, dim=1)[0]

    melanoma_prob = float(probs[1].detach().cpu())
    pred_class = int(melanoma_prob >= threshold)
    pred_prob = float(probs[pred_class].detach().cpu())

    return pred_class, pred_prob, melanoma_prob

class AttentionCapture:
    """
    Captures attention matrices from timm ViT attention dropout modules.
    Requires block.attn.fused_attn = False.
    """
    def __init__(self, model):
        self.model = model
        self.attentions = []
        self.hooks = []

    def _hook_fn(self, module, input, output):
        # output shape: [B, heads, tokens, tokens]
        self.attentions.append(output.detach().cpu())

    def register(self):
        self.remove()
        self.attentions = []

        for block in self.model.blocks:
            if hasattr(block.attn, "fused_attn"):
                block.attn.fused_attn = False

            hook = block.attn.attn_drop.register_forward_hook(self._hook_fn)
            self.hooks.append(hook)

    def remove(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []

def get_vit_attention_maps(model, image_tensor):
    """
    Returns:
    - last_layer_cls_attention: [224,224]
    - attention_rollout: [224,224]
    """
    model.eval()

    for block in model.blocks:
        if hasattr(block.attn, "fused_attn"):
            block.attn.fused_attn = False

    capturer = AttentionCapture(model)
    capturer.register()

    x = image_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        _ = model(x)

    capturer.remove()

    attentions = capturer.attentions

    if len(attentions) == 0:
        raise RuntimeError("No attention maps were captured. Check fused_attn=False and hook location.")

    # Each attention: [1, heads, tokens, tokens]
    # ViT-Tiny patch16_224: tokens = 197 = 1 CLS + 196 patches
    processed_attns = []

    for attn in attentions:
        attn = attn[0]  # [heads, tokens, tokens]
        attn = attn.mean(dim=0)  # [tokens, tokens]
        processed_attns.append(attn)

    num_tokens = processed_attns[0].shape[-1]
    num_patches = num_tokens - 1
    grid_size = int(np.sqrt(num_patches))

    assert grid_size * grid_size == num_patches, f"Unexpected patch count: {num_patches}"

    last_attn = processed_attns[-1]  # [tokens, tokens]
    cls_to_patch = last_attn[0, 1:]  # CLS attends to patches

    last_layer_map = cls_to_patch.reshape(grid_size, grid_size).numpy()
    last_layer_map = cv2.resize(last_layer_map, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    last_layer_map = normalize_map(last_layer_map)

    rollout = torch.eye(num_tokens)

    for attn in processed_attns:
        attn_aug = attn + torch.eye(num_tokens)
        attn_aug = attn_aug / attn_aug.sum(dim=-1, keepdim=True)
        rollout = torch.matmul(attn_aug, rollout)

    rollout_cls = rollout[0, 1:]  # CLS to patches after propagation
    rollout_map = rollout_cls.reshape(grid_size, grid_size).numpy()
    rollout_map = cv2.resize(rollout_map, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    rollout_map = normalize_map(rollout_map)

    del x, attentions, processed_attns
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return last_layer_map, rollout_map

print("Attention helpers ready.")
print("Attention output dir:", ATTN_DIR)


## Attention Extraction Smoke Test


In [ ]:
test_idx = 0

image_tensor, image_np, mask_np, label, image_id = isic_mask_test_ds[test_idx]

pred_before, _, prob_before = predict_vit_image(model_before, image_tensor, threshold=thr_before)
pred_after, _, prob_after = predict_vit_image(model_after, image_tensor, threshold=thr_after)

print("Image:", image_id)
print("True:", CLASS_NAMES[int(label)])
print("Before pred:", CLASS_NAMES[pred_before], "| P(MEL):", prob_before)
print("After pred:", CLASS_NAMES[pred_after], "| P(MEL):", prob_after)

last_before, rollout_before = get_vit_attention_maps(model_before, image_tensor)
last_after, rollout_after = get_vit_attention_maps(model_after, image_tensor)

fig, axes = plt.subplots(1, 6, figsize=(24, 4))

axes[0].imshow(image_np)
axes[0].set_title(f"Original\n{image_id}")
axes[0].axis("off")

axes[1].imshow(mask_np, cmap="gray")
axes[1].set_title("Lesion mask")
axes[1].axis("off")

axes[2].imshow(overlay_heatmap(image_np, last_before, alpha=0.45))
axes[2].set_title("Last-layer attention\nBefore FT")
axes[2].axis("off")

axes[3].imshow(overlay_heatmap(image_np, last_after, alpha=0.45))
axes[3].set_title("Last-layer attention\nAfter FT")
axes[3].axis("off")

axes[4].imshow(overlay_heatmap(image_np, rollout_before, alpha=0.45))
axes[4].set_title("Attention rollout\nBefore FT")
axes[4].axis("off")

axes[5].imshow(overlay_heatmap(image_np, rollout_after, alpha=0.45))
axes[5].set_title("Attention rollout\nAfter FT")
axes[5].axis("off")

plt.tight_layout()
plt.show()


## Test-Set Attention Alignment


In [ ]:
TOP_PERCENT = 20

attention_rows = []
failed_rows = []

stages = {
    "Before FT": {
        "model": model_before,
        "threshold": thr_before
    },
    "After FT": {
        "model": model_after,
        "threshold": thr_after
    }
}

for idx in tqdm(range(len(isic_mask_test_ds)), desc="ViT attention rollout ISIC2017"):
    image_tensor, image_np, mask_np, label, image_id = isic_mask_test_ds[idx]

    true_label = int(label)

    for stage_name, stage_info in stages.items():
        stage_model = stage_info["model"]
        threshold = stage_info["threshold"]

        try:
            pred_class, pred_prob, melanoma_prob = predict_vit_image(
                stage_model,
                image_tensor,
                threshold=threshold
            )

            last_map, rollout_map = get_vit_attention_maps(
                stage_model,
                image_tensor
            )

            maps = {
                "Last-layer CLS attention": last_map,
                "Attention Rollout": rollout_map
            }

            for method_name, attn_map in maps.items():
                metrics = mask_alignment_metrics(
                    attn_map,
                    mask_np,
                    top_percent=TOP_PERCENT
                )

                attention_rows.append({
                    "image_id": image_id,
                    "stage": stage_name,
                    "method": method_name,
                    "true_label": true_label,
                    "true_name": CLASS_NAMES[true_label],
                    "pred_class": pred_class,
                    "pred_name": CLASS_NAMES[pred_class],
                    "pred_prob": pred_prob,
                    "melanoma_prob": melanoma_prob,
                    "threshold": threshold,
                    "top_percent": TOP_PERCENT,
                    "IoU": metrics["IoU"],
                    "Dice": metrics["Dice"],
                    "Inside-Lesion Attribution": metrics["Inside-Lesion Attribution"],
                    "Outside-Lesion Leakage": metrics["Outside-Lesion Leakage"]
                })

        except Exception as e:
            failed_rows.append({
                "image_id": image_id,
                "stage": stage_name,
                "error": str(e)
            })
            print("FAILED:", image_id, stage_name, e)

        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

attention_df = pd.DataFrame(attention_rows)
attention_failed_df = pd.DataFrame(failed_rows)

attention_path = os.path.join(
    ATTN_DIR,
    "table_isic2017_vit_tiny_attention_rollout_per_sample.csv"
)

failed_path = os.path.join(
    ATTN_DIR,
    "table_isic2017_vit_tiny_attention_rollout_failed.csv"
)

attention_df.to_csv(attention_path, index=False)
attention_failed_df.to_csv(failed_path, index=False)

print("Successful rows:", len(attention_df))
print("Failed rows:", len(attention_failed_df))

display(attention_df.head())
display(attention_failed_df.head())

print("Saved:", attention_path)
print("Saved failed:", failed_path)


## Attention Summary Tables


In [ ]:
attention_summary_all = (
    attention_df
    .groupby(["stage", "method"])
    .agg({
        "IoU": "mean",
        "Dice": "mean",
        "Inside-Lesion Attribution": "mean",
        "Outside-Lesion Leakage": "mean"
    })
    .reset_index()
    .sort_values(["stage", "IoU"], ascending=[True, False])
)

attention_summary_melanoma = (
    attention_df[attention_df["true_label"] == 1]
    .groupby(["stage", "method"])
    .agg({
        "IoU": "mean",
        "Dice": "mean",
        "Inside-Lesion Attribution": "mean",
        "Outside-Lesion Leakage": "mean"
    })
    .reset_index()
    .sort_values(["stage", "IoU"], ascending=[True, False])
)

attention_summary_correct_melanoma = (
    attention_df[
        (attention_df["true_label"] == 1) &
        (attention_df["pred_class"] == 1)
    ]
    .groupby(["stage", "method"])
    .agg({
        "IoU": "mean",
        "Dice": "mean",
        "Inside-Lesion Attribution": "mean",
        "Outside-Lesion Leakage": "mean"
    })
    .reset_index()
    .sort_values(["stage", "IoU"], ascending=[True, False])
)

summary_all_path = os.path.join(
    ATTN_DIR,
    "table_isic2017_vit_tiny_attention_summary_all.csv"
)

summary_mel_path = os.path.join(
    ATTN_DIR,
    "table_isic2017_vit_tiny_attention_summary_melanoma.csv"
)

summary_correct_mel_path = os.path.join(
    ATTN_DIR,
    "table_isic2017_vit_tiny_attention_summary_correct_melanoma.csv"
)

attention_summary_all.to_csv(summary_all_path, index=False)
attention_summary_melanoma.to_csv(summary_mel_path, index=False)
attention_summary_correct_melanoma.to_csv(summary_correct_mel_path, index=False)

print("All test samples:")
display(attention_summary_all)

print("Melanoma only:")
display(attention_summary_melanoma)

print("Correct melanoma only:")
display(attention_summary_correct_melanoma)

print("Saved:")
print(summary_all_path)
print(summary_mel_path)
print(summary_correct_mel_path)


## Attention Alignment Comparison


In [ ]:
plot_df = attention_summary_all.copy()

metrics = [
    "IoU",
    "Dice",
    "Inside-Lesion Attribution",
    "Outside-Lesion Leakage"
]

for method_name in plot_df["method"].unique():
    method_df = plot_df[plot_df["method"] == method_name].copy()

    x = np.arange(len(metrics))
    width = 0.35

    before_vals = [
        method_df.loc[method_df["stage"] == "Before FT", metric].values[0]
        for metric in metrics
    ]

    after_vals = [
        method_df.loc[method_df["stage"] == "After FT", metric].values[0]
        for metric in metrics
    ]

    plt.figure(figsize=(10, 5))

    bars1 = plt.bar(x - width/2, before_vals, width, label="Before FT")
    bars2 = plt.bar(x + width/2, after_vals, width, label="After FT")

    plt.xticks(
        x,
        [
            "IoU",
            "Dice",
            "Inside-lesion\nattribution",
            "Outside-lesion\nleakage"
        ]
    )

    plt.ylabel("Mean metric value")
    plt.title(f"ISIC2017 ViT-Tiny: {method_name} before vs after fine-tuning")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)

    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            plt.text(
                bar.get_x() + bar.get_width()/2,
                h,
                f"{h:.3f}",
                ha="center",
                va="bottom",
                fontsize=8,
                rotation=90
            )

    plt.tight_layout()

    safe_method = method_name.replace(" ", "_").replace("-", "_")

    fig_path = os.path.join(
        ATTN_DIR,
        f"fig_isic2017_vit_tiny_{safe_method}_before_after_barplot.png"
    )

    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)


## Melanoma Attention Examples


In [ ]:
# Select melanoma examples
mel_indices = []

for i in range(len(isic_mask_test_ds)):
    _, _, _, label, _ = isic_mask_test_ds[i]

    if int(label) == 1:
        mel_indices.append(i)

# Prefer cases correctly classified after FT
correct_mel_indices = []

for i in mel_indices:
    image_tensor, _, _, label, _ = isic_mask_test_ds[i]
    pred_after, _, _ = predict_vit_image(model_after, image_tensor, threshold=thr_after)

    if pred_after == 1:
        correct_mel_indices.append(i)

if len(correct_mel_indices) >= 3:
    visual_indices = correct_mel_indices[:3]
else:
    visual_indices = mel_indices[:3]

N_VISUAL = len(visual_indices)

print("Visual melanoma cases:", N_VISUAL)

fig, axes = plt.subplots(
    N_VISUAL,
    6,
    figsize=(24, 4.2 * N_VISUAL)
)

if N_VISUAL == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, ds_idx in enumerate(visual_indices):
    image_tensor, image_np, mask_np, label, image_id = isic_mask_test_ds[ds_idx]

    pred_before, _, prob_before = predict_vit_image(model_before, image_tensor, threshold=thr_before)
    pred_after, _, prob_after = predict_vit_image(model_after, image_tensor, threshold=thr_after)

    last_before, rollout_before = get_vit_attention_maps(model_before, image_tensor)
    last_after, rollout_after = get_vit_attention_maps(model_after, image_tensor)

    axes[row_idx, 0].imshow(image_np)
    axes[row_idx, 0].set_title(
        f"Original\n{image_id}\nTrue: melanoma",
        fontsize=9
    )
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(mask_np, cmap="gray")
    axes[row_idx, 1].set_title("Lesion mask", fontsize=9)
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(overlay_heatmap(image_np, last_before, alpha=0.45))
    axes[row_idx, 2].set_title(
        f"Last attention\nBefore FT\nPred: {CLASS_NAMES[pred_before]}\nP(MEL)={prob_before:.3f}",
        fontsize=8
    )
    axes[row_idx, 2].axis("off")

    axes[row_idx, 3].imshow(overlay_heatmap(image_np, last_after, alpha=0.45))
    axes[row_idx, 3].set_title(
        f"Last attention\nAfter FT\nPred: {CLASS_NAMES[pred_after]}\nP(MEL)={prob_after:.3f}",
        fontsize=8
    )
    axes[row_idx, 3].axis("off")

    axes[row_idx, 4].imshow(overlay_heatmap(image_np, rollout_before, alpha=0.45))
    axes[row_idx, 4].set_title("Attention rollout\nBefore FT", fontsize=8)
    axes[row_idx, 4].axis("off")

    axes[row_idx, 5].imshow(overlay_heatmap(image_np, rollout_after, alpha=0.45))
    axes[row_idx, 5].set_title("Attention rollout\nAfter FT", fontsize=8)
    axes[row_idx, 5].axis("off")

plt.suptitle(
    "ISIC2017 ViT-Tiny attention visualization and attention rollout",
    fontsize=16,
    y=1.01
)

plt.tight_layout()

visual_path = os.path.join(
    ATTN_DIR,
    "fig_isic2017_vit_tiny_attention_rollout_melanoma_examples.png"
)

plt.savefig(visual_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", visual_path)


## Paired Attention Improvement Analysis


In [ ]:
!pip -q install scipy

import os
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

metrics = [
    "IoU",
    "Dice",
    "Inside-Lesion Attribution",
    "Outside-Lesion Leakage"
]

paired_rows = []

for method_name in attention_df["method"].unique():
    method_df = attention_df[attention_df["method"] == method_name].copy()

    for metric in metrics:
        pivot = method_df.pivot_table(
            index="image_id",
            columns="stage",
            values=metric,
            aggfunc="mean"
        ).dropna()

        if "Before FT" not in pivot.columns or "After FT" not in pivot.columns:
            continue

        before = pivot["Before FT"].values
        after = pivot["After FT"].values

        delta = after - before

        # For Outside-Lesion Leakage, lower is better
        if metric == "Outside-Lesion Leakage":
            improvement = before - after
            percent_improved = np.mean(after < before) * 100
        else:
            improvement = after - before
            percent_improved = np.mean(after > before) * 100

        try:
            stat, p_value = wilcoxon(improvement)
        except Exception:
            stat, p_value = np.nan, np.nan

        cohen_dz = np.mean(improvement) / (np.std(improvement) + 1e-8)

        paired_rows.append({
            "method": method_name,
            "metric": metric,
            "n_pairs": len(pivot),
            "mean_before": float(np.mean(before)),
            "mean_after": float(np.mean(after)),
            "mean_delta_after_minus_before": float(np.mean(delta)),
            "median_delta_after_minus_before": float(np.median(delta)),
            "percent_improved": float(percent_improved),
            "wilcoxon_statistic": float(stat) if not np.isnan(stat) else np.nan,
            "wilcoxon_p_value": float(p_value) if not np.isnan(p_value) else np.nan,
            "cohen_dz_improvement": float(cohen_dz)
        })

paired_attention_stats_df = pd.DataFrame(paired_rows)

paired_attention_stats_path = os.path.join(
    ATTN_DIR,
    "table_isic2017_vit_tiny_paired_attention_improvement_stats.csv"
)

paired_attention_stats_df.to_csv(paired_attention_stats_path, index=False)

display(paired_attention_stats_df)

print("Saved:", paired_attention_stats_path)


## Paired Analysis for Melanoma Samples


In [ ]:
mel_attention_df = attention_df[attention_df["true_label"] == 1].copy()

mel_paired_rows = []

for method_name in mel_attention_df["method"].unique():
    method_df = mel_attention_df[mel_attention_df["method"] == method_name].copy()

    for metric in metrics:
        pivot = method_df.pivot_table(
            index="image_id",
            columns="stage",
            values=metric,
            aggfunc="mean"
        ).dropna()

        if "Before FT" not in pivot.columns or "After FT" not in pivot.columns:
            continue

        before = pivot["Before FT"].values
        after = pivot["After FT"].values

        delta = after - before

        if metric == "Outside-Lesion Leakage":
            improvement = before - after
            percent_improved = np.mean(after < before) * 100
        else:
            improvement = after - before
            percent_improved = np.mean(after > before) * 100

        try:
            stat, p_value = wilcoxon(improvement)
        except Exception:
            stat, p_value = np.nan, np.nan

        cohen_dz = np.mean(improvement) / (np.std(improvement) + 1e-8)

        mel_paired_rows.append({
            "method": method_name,
            "metric": metric,
            "n_pairs": len(pivot),
            "mean_before": float(np.mean(before)),
            "mean_after": float(np.mean(after)),
            "mean_delta_after_minus_before": float(np.mean(delta)),
            "median_delta_after_minus_before": float(np.median(delta)),
            "percent_improved": float(percent_improved),
            "wilcoxon_statistic": float(stat) if not np.isnan(stat) else np.nan,
            "wilcoxon_p_value": float(p_value) if not np.isnan(p_value) else np.nan,
            "cohen_dz_improvement": float(cohen_dz)
        })

mel_paired_attention_stats_df = pd.DataFrame(mel_paired_rows)

mel_paired_attention_stats_path = os.path.join(
    ATTN_DIR,
    "table_isic2017_vit_tiny_paired_attention_improvement_stats_melanoma.csv"
)

mel_paired_attention_stats_df.to_csv(mel_paired_attention_stats_path, index=False)

display(mel_paired_attention_stats_df)

print("Saved:", mel_paired_attention_stats_path)


## Prediction Transitions and Attention Change


In [ ]:
# Build per-image prediction transition table from attention_df
pred_stage_df = (
    attention_df
    .groupby(["image_id", "stage"])
    .agg({
        "true_label": "first",
        "true_name": "first",
        "pred_class": "first",
        "pred_name": "first",
        "melanoma_prob": "first"
    })
    .reset_index()
)

pred_pivot = pred_stage_df.pivot(
    index="image_id",
    columns="stage",
    values=["true_label", "true_name", "pred_class", "pred_name", "melanoma_prob"]
)

pred_pivot.columns = [
    f"{a}_{b.replace(' ', '_')}"
    for a, b in pred_pivot.columns
]

pred_pivot = pred_pivot.reset_index()

def transition_label(row):
    true_label = int(row["true_label_Before_FT"])
    pred_before = int(row["pred_class_Before_FT"])
    pred_after = int(row["pred_class_After_FT"])

    if true_label == 1:
        if pred_before == 0 and pred_after == 1:
            return "FN_to_TP_melanoma"
        elif pred_before == 1 and pred_after == 1:
            return "TP_to_TP_melanoma"
        elif pred_before == 0 and pred_after == 0:
            return "FN_to_FN_melanoma"
        elif pred_before == 1 and pred_after == 0:
            return "TP_to_FN_melanoma"

    if true_label == 0:
        if pred_before == 0 and pred_after == 0:
            return "TN_to_TN_non_melanoma"
        elif pred_before == 1 and pred_after == 0:
            return "FP_to_TN_non_melanoma"
        elif pred_before == 1 and pred_after == 1:
            return "FP_to_FP_non_melanoma"
        elif pred_before == 0 and pred_after == 1:
            return "TN_to_FP_non_melanoma"

    return "other"

pred_pivot["transition"] = pred_pivot.apply(transition_label, axis=1)

pred_pivot["delta_prob_melanoma"] = (
    pred_pivot["melanoma_prob_After_FT"].astype(float)
    - pred_pivot["melanoma_prob_Before_FT"].astype(float)
)

transition_counts = pred_pivot["transition"].value_counts().reset_index()
transition_counts.columns = ["transition", "count"]

display(transition_counts)

transition_path = os.path.join(
    ATTN_DIR,
    "table_isic2017_vit_tiny_prediction_transitions.csv"
)

pred_pivot.to_csv(transition_path, index=False)

print("Saved:", transition_path)


## Attention Improvement by Prediction Transition


In [ ]:
# Create paired attention metric deltas
delta_rows = []

for method_name in attention_df["method"].unique():
    method_df = attention_df[attention_df["method"] == method_name].copy()

    for metric in metrics:
        pivot = method_df.pivot_table(
            index="image_id",
            columns="stage",
            values=metric,
            aggfunc="mean"
        ).dropna()

        if "Before FT" not in pivot.columns or "After FT" not in pivot.columns:
            continue

        temp = pivot.reset_index()
        temp["method"] = method_name
        temp["metric"] = metric
        temp["before"] = temp["Before FT"]
        temp["after"] = temp["After FT"]
        temp["delta_after_minus_before"] = temp["after"] - temp["before"]

        if metric == "Outside-Lesion Leakage":
            temp["improvement"] = temp["before"] - temp["after"]
        else:
            temp["improvement"] = temp["after"] - temp["before"]

        delta_rows.append(
            temp[[
                "image_id",
                "method",
                "metric",
                "before",
                "after",
                "delta_after_minus_before",
                "improvement"
            ]]
        )

attention_delta_df = pd.concat(delta_rows, ignore_index=True)

attention_delta_df = attention_delta_df.merge(
    pred_pivot[["image_id", "transition", "delta_prob_melanoma"]],
    on="image_id",
    how="left"
)

transition_attention_summary = (
    attention_delta_df
    .groupby(["transition", "method", "metric"])
    .agg({
        "before": "mean",
        "after": "mean",
        "delta_after_minus_before": "mean",
        "improvement": "mean",
        "image_id": "count",
        "delta_prob_melanoma": "mean"
    })
    .reset_index()
    .rename(columns={"image_id": "n"})
    .sort_values(["transition", "method", "metric"])
)

transition_attention_path = os.path.join(
    ATTN_DIR,
    "table_isic2017_vit_tiny_attention_improvement_by_prediction_transition.csv"
)

transition_attention_summary.to_csv(transition_attention_path, index=False)

display(transition_attention_summary)

print("Saved:", transition_attention_path)
